# Phase 3, experiment 9: graph score smoothing

**Why this experiment:** `05_weak_steps.ipynb` found that the illicit transactions the model misses
come in connected groups: 81% of their neighbours are also illicit. Adding the neighbours' raw
features as columns did not help (notebook 06). This experiment uses the neighbours' **predicted
scores** instead. A predicted score already sums up "how illicit does this transaction look", so if
the model catches some members of a group, their neighbours can be pulled up with them.

**The one change in this experiment:** after the model has predicted every transaction in the period,
each transaction's score is blended with its neighbours' scores:

`smoothed score = (1 - w) x own score + w x neighbour score`

- `w` (the blending weight) is between 0 and 1. `w = 0` means no smoothing (the plain model).
- The neighbour score is either the **mean** or the **max** of the neighbours' predicted scores.
  Mean asks "do my neighbours look illicit on average?"; max asks "does any neighbour look illicit?".
- Two scales are tried. On the **probability** scale the model's own scores are blended directly.
  LightGBM's scores are very extreme (most are within 0.001 of 0 or 1), so even a small weight moves a
  licit transaction with an illicit neighbour far up the ranking. On the **rank** scale each score is
  first replaced by its position in the period (divided by the number of transactions, so it runs from
  0 to 1), which makes the blend much gentler.
- Neighbours in both directions (predecessors and successors) are used. Neighbours outside train and
  test have no prediction and are skipped. A transaction with no scored neighbour keeps its own score.

**No labels are used for smoothing.** The neighbour scores are the model's predictions, which exist
for labeled, unlabeled and test rows alike, so this can be applied to the test set exactly as it is
applied to the validation folds. Edges never cross time steps, so a validation transaction's
neighbours are all in the same validation period.

**Decision:** try graph score smoothing first, then recent-step weighting, and stop
experimenting by about 17 Oct.

Model: default LightGBM on the 165 raw features. Validation: the three expanding-window folds.

## 1. Load

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

SEED = 42
DATA_DIR = "../data"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
edges = pd.read_csv(f"{DATA_DIR}/txs_edgelist.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

raw_cols = [c for c in train.columns if c.startswith("feat_")]

labeled = train[train["label"].notna()].copy()
labeled["label"] = labeled["label"].astype(int)

## 2. The smoothing function

Each edge is listed in both directions, so that for every transaction we can collect all of its
neighbours, whether they come before or after it.

In [ ]:
# Every edge in both directions: (txId, neighbour)
both_directions = pd.concat([
    edges.rename(columns={"txId1": "txId", "txId2": "neighbour"}),
    edges.rename(columns={"txId2": "txId", "txId1": "neighbour"}),
], ignore_index=True)

def smooth_scores(scores, method, weight):
    # scores: a Series of predicted scores indexed by txId (every transaction in the period).
    # method: "mean" or "max" of the neighbours' scores.
    # weight: how much of the neighbour score to blend in (0 = no smoothing).
    # Returns a Series of smoothed scores with the same index.
    if weight == 0:
        return scores

    # Attach each neighbour's score
    # Neighbours without a score get NaN and are dropped
    pairs = both_directions[both_directions["txId"].isin(scores.index)].copy()
    pairs["neighbour_score"] = pairs["neighbour"].map(scores)
    pairs = pairs.dropna(subset=["neighbour_score"])

    # One neighbour score per transaction: the mean or the max over its neighbours
    neighbour_score = pairs.groupby("txId")["neighbour_score"].agg(method)

    # Transactions with no scored neighbour keep their own score
    neighbour_score = neighbour_score.reindex(scores.index).fillna(scores)

    return (1 - weight) * scores + weight * neighbour_score

## 3. Score every smoothing option on the folds

For each fold: train the default LightGBM on the fold's training steps (labeled rows only), predict
**every** transaction in the validation steps (labeled and unlabeled, because most neighbours are
unlabeled), smooth, then compute the AUC on the labeled validation rows. Smoothing needs no
retraining, so all options reuse the same predictions.

In [ ]:
FOLDS = [
    {"name": "fold 1", "train_last": 14, "valid_last": 21},
    {"name": "fold 2", "train_last": 21, "valid_last": 28},
    {"name": "fold 3", "train_last": 28, "valid_last": 35},
]
fold_names = [f["name"] for f in FOLDS]

# (scale, method, weight) combinations
OPTIONS = [("none", "none", 0.0)]
for scale in ["probability", "rank"]:
    for method in ["mean", "max"]:
        for weight in [0.01, 0.05, 0.1, 0.2]:
            OPTIONS.append((scale, method, weight))

def option_label(scale, method, weight):
    return "none" if weight == 0 else f"{scale}, {method}, w={weight}"

score_rows = []
step24_rows = []

for fold in FOLDS:
    # Training rows
    tr = labeled[labeled["time_step"] <= fold["train_last"]]
    # Every transaction in the validation period, labeled or not
    period = train[(train["time_step"] > fold["train_last"]) & (train["time_step"] <= fold["valid_last"])]

    model = lgb.LGBMClassifier(random_state=SEED, verbose=-1)
    model.fit(tr[raw_cols], tr["label"])
    period_scores = pd.Series(model.predict_proba(period[raw_cols])[:, 1], index=period["txId"].values)
    # The same scores on the rank scale: position in the period from 0 to 1
    period_ranks = period_scores.rank() / len(period_scores)

    # Labeled rows of the validation period, used to compute the AUC
    va = period[period["label"].notna()]
    va_label = va["label"].astype(int).values

    for scale, method, weight in OPTIONS:
        base = period_ranks if scale == "rank" else period_scores
        smoothed = smooth_scores(base, method, weight)
        va_pred = smoothed.loc[va["txId"]].values
        option_name = option_label(scale, method, weight)
        score_rows.append({"option": option_name, "fold": fold["name"],
                           "val_auc": roc_auc_score(va_label, va_pred)})

        if fold["name"] == "fold 2":
            in_24 = (va["time_step"] == 24).values
            step24_rows.append({"option": option_name, "auc_step_24": roc_auc_score(va_label[in_24], va_pred[in_24])})
    print("done:", fold["name"])

done: fold 1
done: fold 2
done: fold 3


## 4. Results

`change_mean` and `change_fold2` compare each option with no smoothing (first row). As in notebook 10,
an option is worth keeping only if it raises the mean without lowering fold 2.

In [4]:
option_order = [option_label(sc, m, w) for sc, m, w in OPTIONS]
results = pd.DataFrame(score_rows).pivot(index="option", columns="fold", values="val_auc").loc[option_order]
results["mean"] = results[fold_names].mean(axis=1)
results["change_mean"] = results["mean"] - results.loc["none", "mean"]
results["change_fold2"] = results["fold 2"] - results.loc["none", "fold 2"]
results = results.join(pd.DataFrame(step24_rows).set_index("option"))
results.round(4)

,fold 1,fold 2,fold 3,mean,change_mean,change_fold2,auc_step_24
option,,,,,,,
none,0.9888,0.9572,0.9965,0.9808,0.0000,0.0000,0.7866
"probability, mean, w=0.01",0.9862,0.9333,0.9963,0.9719,-0.0089,-0.0240,0.7136
"probability, mean, w=0.05",0.9778,0.9272,0.9952,0.9668,-0.0141,-0.0300,0.6870
"probability, mean, w=0.1",0.9731,0.9236,0.9946,0.9638,-0.0170,-0.0336,0.6721
"probability, mean, w=0.2",0.9675,0.9200,0.9937,0.9604,-0.0204,-0.0372,0.6586
"probability, max, w=0.01",0.9840,0.9310,0.9957,0.9702,-0.0106,-0.0262,0.7039
"probability, max, w=0.05",0.9675,0.9237,0.9944,0.9619,-0.0189,-0.0335,0.6707
"probability, max, w=0.1",0.9624,0.9204,0.9937,0.9588,-0.0220,-0.0368,0.6586
"probability, max, w=0.2",0.9568,0.9170,0.9928,0.9555,-0.0253,-0.0402,0.6506


## 5. No submission

No smoothing option raised the fold mean, so no submission is made from this notebook. The
`smooth_scores` function can be applied to test predictions in the same way if a later model needs it.

## Findings

- **Does smoothing raise the fold mean without lowering fold 2?** No. None of the 16 options raises the
  mean, and all 16 lower fold 2. The best options (rank scale, weight 0.01) leave the mean unchanged
  (0.9808) and lower fold 2 by 0.0002 to 0.0003, which is the same as not smoothing at all.
- **Probability scale or rank scale?** The probability scale hurts at every weight, even 0.01: the mean
  falls by 0.009 to 0.025 and fold 2 by 0.024 to 0.040. LightGBM's scores are very extreme, so even a
  small share of an illicit neighbour's score (close to 1) lifts a licit transaction above illicit
  transactions that the model scored close to 0. The rank scale is gentler: roughly neutral at weights
  of 0.01 to 0.05 (mean -0.0000 to -0.0005) and harmful at 0.2 (mean -0.0047 for mean, -0.0071 for max).
- **Mean or max, and which weight?** Mean is slightly less harmful than max at every weight and scale,
  and smaller weights are always less harmful. No weight is better than no smoothing.
- **What happens on step 24?** Every option lowers it, from 0.787 without smoothing to between 0.651 and
  0.786. This fits notebook 05: 78% of the missed illicit transactions' neighbours were also missed, so
  in most groups there is no high-scoring neighbour to pull them up, while licit neighbours of illicit
  transactions do get pulled up.
- **Decision: use smoothing in the final model?** No. No option raised the fold mean, and all of them
  lowered fold 2.
